## Deep Learning for Mortality Prediction (DLMP)

### Import packages 

In [1]:
import tensorflow as tf
import numpy as np
import os as os
tfkl = tf.keras.layers
import random

### Import functions

In [7]:
import training_functions_uncertainty as training_functions
import importlib

importlib.reload(training_functions)

<module 'training_functions_uncertainty' from '/Users/paigepark/Desktop/repos/deep-mort/code/training_functions_uncertainty.py'>

### Import data

#### State data

In [5]:
state_training = np.loadtxt('../data/state_training.txt')
state_test = np.loadtxt('../data/state_test.txt')

#### Country data

In [21]:
country_training = np.loadtxt('../data/country_training.txt')
country_test = np.loadtxt('../data/country_test.txt')

In [22]:
country_training.shape

(359594, 5)

#### Combined data

In [5]:
combined_training = np.loadtxt('../data/combined_training.txt')
combined_test = np.loadtxt('../data/combined_test.txt')

In [19]:
geos_key = np.load('../data/geos_key.npy')

In [20]:
geos_key

array([['AK', '0'],
       ['AL', '1'],
       ['AZ', '2'],
       ['AR', '3'],
       ['CA', '4'],
       ['CO', '5'],
       ['CT', '6'],
       ['DE', '7'],
       ['FL', '8'],
       ['GA', '9'],
       ['HI', '10'],
       ['ID', '11'],
       ['IL', '12'],
       ['IN', '13'],
       ['IA', '14'],
       ['KS', '15'],
       ['KY', '16'],
       ['LA', '17'],
       ['ME', '18'],
       ['MD', '19'],
       ['MA', '20'],
       ['MI', '21'],
       ['MN', '22'],
       ['MS', '23'],
       ['MO', '24'],
       ['MT', '25'],
       ['NE', '26'],
       ['NV', '27'],
       ['NH', '28'],
       ['NJ', '29'],
       ['NM', '30'],
       ['NY', '31'],
       ['NC', '32'],
       ['ND', '33'],
       ['OH', '34'],
       ['OK', '35'],
       ['OR', '36'],
       ['PA', '37'],
       ['RI', '38'],
       ['SC', '39'],
       ['SD', '40'],
       ['TN', '41'],
       ['TX', '42'],
       ['UT', '43'],
       ['VT', '44'],
       ['VA', '45'],
       ['WA', '46'],
       ['WV', '47'],
  

In [ ]:
geo_dict = {int(code): geo for geo, code in geos_key}

#### All Country Model

These models are those used in the paper to produce all of main figures/table in the paper.

In [23]:
# prep data
country_train_prepped = training_functions.prep_data(country_training, mode="train", changeratetolog=True)
country_test_prepped = training_functions.prep_data(country_test, mode="test", changeratetolog=True)

In [ ]:
# get the proper geography input dimension for model set up 
unique_vals = tf.unique(country_training[:, 0]).y
country_geo_dim = np.array(tf.size(unique_vals)).item()
country_geo_dim = country_geo_dim + 50

tf.Tensor(
[50. 51. 52. 53. 54. 55. 56. 57. 58. 59. 60. 61. 62. 63. 64. 65. 66. 67.
 68. 69. 70. 71. 72. 73. 74. 75. 76. 77. 78. 79. 80. 81. 82. 83. 84. 85.
 86. 87. 88. 89.], shape=(40,), dtype=float64)
90


In [25]:
# Prepare input features once (shared across all iterations)
training_input_features = (
    tf.convert_to_tensor((country_training[:, 2] - 1959) / 60, dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 3], dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 0], dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 1], dtype=tf.float32),
)

test_input_features = (
    tf.convert_to_tensor((country_test[:, 2] - 1959) / 60, dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 3], dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 0], dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 1], dtype=tf.float32),
)

inputs_train = np.delete(country_training, 4, axis=1)
inputs_test = np.delete(country_test, 4, axis=1)

# Storage for ensemble combination
M = 5
train_mus = []
train_vars = []
test_mus = []
test_vars = []

In [ ]:
# train each ensemble member with different seeds, saving predictions and models per member
for i in range(1, M + 1):
    # Set reproducible seeds per iteration
    np.random.seed(i)
    tf.random.set_seed(i)
    random.seed(i)
    os.environ['PYTHONHASHSEED'] = str(i)

    model_country, loss_info_country = training_functions.run_deep_ensemble_model(
        country_train_prepped, country_test_prepped, country_geo_dim,
        epochs=20, steps_per_epoch=1405, lograte=True
    )

    # Get raw predictions: shape (N, 2) = [mu, raw_variance]
    raw_train_preds = model_country.predict(training_input_features)
    raw_test_preds = model_country.predict(test_input_features)

    # Split into mu and variance (applying softplus + floor to raw variance)
    train_mu = raw_train_preds[:, 0:1]
    train_var = np.log(1.0 + np.exp(raw_train_preds[:, 1:2])) + 1e-6

    test_mu = raw_test_preds[:, 0:1]
    test_var = np.log(1.0 + np.exp(raw_test_preds[:, 1:2])) + 1e-6

    # Store for ensemble combination
    train_mus.append(train_mu)
    train_vars.append(train_var)
    test_mus.append(test_mu)
    test_vars.append(test_var)

    # Save individual member predictions: [geo, gender, year, age, mu, variance]
    training_predictions = np.column_stack((inputs_train, train_mu, train_var))
    test_predictions = np.column_stack((inputs_test, test_mu, test_var))

    model_country.save(f"../models/dl_country_ensemble_{i}.keras")
    np.savetxt(f"../data/dl_country_fitted_{i}.txt", training_predictions)
    np.savetxt(f"../data/dl_country_forecast_{i}.txt", test_predictions)

    print(f"Ensemble member {i} complete | val NLL: {loss_info_country:.6f}")

Epoch 1/20
1405/1405 - 6s - 4ms/step - loss: 1.8986 - val_loss: -1.5452e-01 - learning_rate: 1.0000e-03
Epoch 2/20
1405/1405 - 5s - 3ms/step - loss: -1.9042e-01 - val_loss: -5.0848e-01 - learning_rate: 1.0000e-03
Epoch 3/20
1405/1405 - 5s - 3ms/step - loss: -4.9508e-01 - val_loss: -6.5366e-01 - learning_rate: 1.0000e-03
Epoch 4/20
1405/1405 - 4s - 3ms/step - loss: -6.1049e-01 - val_loss: -5.5770e-01 - learning_rate: 1.0000e-03
Epoch 5/20
1405/1405 - 5s - 3ms/step - loss: -6.8999e-01 - val_loss: -8.2522e-01 - learning_rate: 1.0000e-03
Epoch 6/20
1405/1405 - 5s - 3ms/step - loss: -7.7028e-01 - val_loss: -8.2215e-01 - learning_rate: 1.0000e-03
Epoch 7/20
1405/1405 - 5s - 3ms/step - loss: -8.3801e-01 - val_loss: -8.2355e-01 - learning_rate: 1.0000e-03
Epoch 8/20
1405/1405 - 4s - 3ms/step - loss: -9.0119e-01 - val_loss: -9.7221e-01 - learning_rate: 1.0000e-03
Epoch 9/20
1405/1405 - 4s - 3ms/step - loss: -9.3611e-01 - val_loss: -9.5972e-01 - learning_rate: 1.0000e-03
Epoch 10/20
1405/1405 - 

In [27]:
# combine ensemble members to get overall predictions and uncertainty decomposition
train_mus = np.array(train_mus)    # (M, N, 1)
train_vars = np.array(train_vars)  # (M, N, 1)
test_mus = np.array(test_mus)      # (M, N, 1)
test_vars = np.array(test_vars)    # (M, N, 1)

# Ensemble mean: average of individual means
ensemble_train_mu = np.mean(train_mus, axis=0)
ensemble_test_mu = np.mean(test_mus, axis=0)

# Ensemble variance: (1/M) * sum(sigma_m^2 + mu_m^2) - mu*^2
ensemble_train_var = np.mean(train_vars + train_mus**2, axis=0) - ensemble_train_mu**2
ensemble_test_var = np.mean(test_vars + test_mus**2, axis=0) - ensemble_test_mu**2

# Decompose into aleatoric (data noise) and epistemic (model disagreement)
aleatoric_train = np.mean(train_vars, axis=0)
epistemic_train = np.mean(train_mus**2, axis=0) - ensemble_train_mu**2

aleatoric_test = np.mean(test_vars, axis=0)
epistemic_test = np.mean(test_mus**2, axis=0) - ensemble_test_mu**2

# 95% prediction intervals
z = 1.96
ensemble_train_std = np.sqrt(ensemble_train_var)
ensemble_test_std = np.sqrt(ensemble_test_var)

train_lower = ensemble_train_mu - z * ensemble_train_std
train_upper = ensemble_train_mu + z * ensemble_train_std
test_lower = ensemble_test_mu - z * ensemble_test_std
test_upper = ensemble_test_mu + z * ensemble_test_std

In [ ]:
# Save: [geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95]
ensemble_train_output = np.column_stack((
    inputs_train, ensemble_train_mu, ensemble_train_var,
    aleatoric_train, epistemic_train, train_lower, train_upper
))
ensemble_test_output = np.column_stack((
    inputs_test, ensemble_test_mu, ensemble_test_var,
    aleatoric_test, epistemic_test, test_lower, test_upper
))

np.savetxt("../data/dl_country_ensemble_fitted.txt", ensemble_train_output)
np.savetxt("../data/dl_country_ensemble_forecast.txt", ensemble_test_output)

print("\nEnsemble combination complete.")
print(f"Output columns: geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95")
print(f"Train shape: {ensemble_train_output.shape}")
print(f"Test shape:  {ensemble_test_output.shape}")

NameError: name 'np' is not defined